# British Airways Booking Prediction

This notebook builds a machine learning model to predict whether a customer completes a booking using booking behavior, trip details, and ancillary preferences.

## 1. Import libraries
Import the required packages for data handling, visualization, preprocessing, resampling, modeling, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('viridis')

## 2. Load the dataset
Load the customer booking dataset. Update the file path if your CSV is stored in a different location.

In [ ]:
# Replace the path below if needed
df = pd.read_csv('/Users/utkarsh/Downloads/customer_booking.csv', encoding='latin-1')

print('Shape:', df.shape)
print('
Columns:')
print(df.columns.tolist())
print('
Target distribution:')
print(df['booking_complete'].value_counts(normalize=True))
print('
Missing values:')
print(df.isnull().sum())

df.head()

## 3. Feature engineering
Create a few business-driven features to capture customer intent and travel behavior more effectively.

In [ ]:
# Trip complexity combines group size and stay length
df['trip_complexity'] = df['num_passengers'] * df['length_of_stay']

# Premium score counts how many paid add-ons a customer selected
df['premium_score'] = (
    df['wants_extra_baggage'] +
    df['wants_preferred_seat'] +
    df['wants_in_flight_meals']
)

# Bucket lead time into interpretable booking timing groups
df['booking_timing'] = pd.cut(
    df['purchase_lead'],
    bins=[0, 30, 90, 180, np.inf],
    labels=['LastMin', 'Short', 'Medium', 'Long']
)

# Encode flight hour cyclically to preserve time-of-day continuity
df['flight_hour_sin'] = np.sin(2 * np.pi * df['flight_hour'] / 24)
df['flight_hour_cos'] = np.cos(2 * np.pi * df['flight_hour'] / 24)

df[['trip_complexity', 'premium_score', 'booking_timing', 'flight_hour_sin', 'flight_hour_cos']].head()

## 4. Encode categorical features
Machine learning models and SMOTE require numeric inputs, so categorical variables are label-encoded here.

In [ ]:
cat_cols = [
    'sales_channel',
    'trip_type',
    'route',
    'booking_origin',
    'booking_timing',
    'flight_day'
]

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print('Encoded columns created:')
print([col + '_enc' for col in cat_cols])

## 5. Prepare features and target
Split the dataset into numerical, binary, and encoded categorical features.

In [ ]:
num_cols = [
    'num_passengers', 'purchase_lead', 'length_of_stay',
    'flight_hour', 'flight_duration',
    'trip_complexity', 'premium_score',
    'flight_hour_sin', 'flight_hour_cos'
]

binary_cols = [
    'wants_extra_baggage',
    'wants_preferred_seat',
    'wants_in_flight_meals'
]

enc_cols = [c + '_enc' for c in cat_cols]

feature_cols = num_cols + binary_cols + enc_cols

X = df[feature_cols].copy()
y = df['booking_complete']

print('Total features:', len(feature_cols))
print(feature_cols)

## 6. Train-test split and scaling
Scale only the continuous numerical variables. Binary and encoded features are left unchanged.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)

## 7. Handle class imbalance with SMOTE
The target is imbalanced, so SMOTE is used on the training set to improve minority-class learning.

In [ ]:
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('Original training target distribution:')
print(y_train.value_counts())
print('
Balanced training target distribution:')
print(pd.Series(y_train_bal).value_counts())

## 8. Train the Random Forest model
Fit a Random Forest classifier using the balanced training data.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_bal, y_train_bal)

## 9. Evaluate model performance
Assess the model using ROC-AUC, classification report, cross-validation, confusion matrix, and ROC curve.

In [ ]:
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print('AUC-ROC:', round(auc, 3))
print('
Classification Report:')
print(classification_report(y_test, y_pred))

cv_scores = cross_val_score(rf, X_train_bal, y_train_bal, cv=5, scoring='roc_auc')
print(f'CV AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Feature importance
Inspect which variables contribute most to the model's decision-making.

In [ ]:
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importances.head(10), x='importance', y='feature', palette='viridis')
plt.title('Top 10 Feature Importances - Booking Prediction')
plt.tight_layout()
plt.show()

importances.head(10)

## 11. Key findings
- The model delivers strong discrimination performance using ROC-AUC.
- Purchase lead time, route, trip complexity, and premium intent are among the strongest predictors.
- The workflow is suitable for proactive marketing and customer targeting use cases.